# Clinical GenAI RAG System

Retrieval-Augmented Generation system for clinical question answering using MIMIC clinical data.

## 1. Install Required Libraries

In [ ]:
!pip install -q pandas faiss-cpu langchain openai transformers sentence-transformers accelerate bitsandbytes

## 2. Mount Google Drive and Set Data Path

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

data_path = '/content/drive/MyDrive/clinical_data'

print('Data path:', data_path)

## 3. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import faiss
import torch
import re
import matplotlib.pyplot as plt

from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    BitsAndBytesConfig
)

print('Libraries imported successfully.')

## 4. Load Clinical Data

In [ ]:
admissions = pd.read_csv(os.path.join(data_path, 'ADMISSIONS.csv'))
patients = pd.read_csv(os.path.join(data_path, 'PATIENTS.csv'))
labitems = pd.read_csv(os.path.join(data_path, 'D_LABITEMS.csv'))
structured = pd.read_csv(os.path.join(data_path, 'structured_medical_records.csv'))
labevents = pd.read_csv(os.path.join(data_path, 'LABEVENTS.csv'), nrows=100000)

print('Admissions:', admissions.shape)
print('Patients:', patients.shape)
print('Lab Items:', labitems.shape)
print('Structured Records:', structured.shape)
print('Lab Events:', labevents.shape)

## 5. Inspect Clinical Data

In [ ]:
print('Structured columns:')
print(structured.columns.tolist())

print('\nAdmissions columns:')
print(admissions.columns.tolist())

print('\nFirst five structured records:')
display(structured.head())

## 6. Prepare Clinical Documents

In [ ]:
documents = []

for _, row in structured.iterrows():
    values = []

    for column in structured.columns:
        if pd.notna(row[column]):
            values.append(f'{column}: {row[column]}')

    documents.append(' | '.join(values))

print('Number of clinical documents:', len(documents))
print('\nExample document:')
print(documents[0] if documents else 'No documents found.')

## 7. Create Sentence Transformer Embeddings

In [ ]:
embedding_model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

print('Creating embeddings...')

embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True,
    convert_to_numpy=True
).astype('float32')

print('Embedding shape:', embeddings.shape)

## 8. Create FAISS Vector Index

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print('FAISS index created successfully.')
print('Number of indexed documents:', index.ntotal)

## 9. Load Mistral-7B-Instruct Model

In [ ]:
model_id = 'mistralai/Mistral-7B-Instruct-v0.3'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto'
)

generator = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150
)

print('Mistral model loaded successfully.')

## 10. Define the RAG Function

In [ ]:
def run_rag(question, k=3):

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype('float32')

    distances, indices = index.search(
        question_embedding,
        k=k
    )

    retrieved_documents = [
        documents[i] for i in indices[0]
    ]

    context = '\n---\n'.join(retrieved_documents)

    prompt = f'''<s>[INST]
Use the following clinical information to answer the question.

Clinical Information:
{context}

Question:
{question}

Answer using only the available clinical information.
[/INST]'''

    output = generator(
        prompt,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    answer = output[0]['generated_text']
    answer = answer.split('[/INST]')[-1].strip()

    return answer, retrieved_documents

## 11. Test the RAG System

In [ ]:
question = 'What is the diagnosis information for the patient?'

answer, retrieved_context = run_rag(question)

print('Question:')
print(question)

print('\nRetrieved Clinical Context:')

for i, document in enumerate(retrieved_context, 1):
    print(f'\nDocument {i}:')
    print(document)

print('\nGenerated Answer:')
print(answer)

## 12. Load Clinical QA Test Dataset

In [ ]:
question_file = '/content/drive/MyDrive/mimic_iii_qa_testset.csv'

if os.path.exists(question_file):
    questions_df = pd.read_csv(question_file)

    print('Question dataset loaded successfully.')
    print('Number of questions:', len(questions_df))
    display(questions_df.head())
else:
    questions_df = None
    print('Question dataset not found.')

## 13. Evaluate RAG Results

In [ ]:
results = []

if questions_df is not None:

    total_questions = min(25, len(questions_df))

    for i in range(total_questions):

        question = questions_df.iloc[i]['Question_Text']

        try:
            answer, _ = run_rag(question)

            results.append({
                'ID': i,
                'Question': question,
                'Answer': answer
            })

            print(f'Processed {i + 1}/{total_questions}')

        except Exception as e:
            print(f'Error on question {i + 1}: {e}')

            results.append({
                'ID': i,
                'Question': question,
                'Answer': 'Error'
            })

    results_df = pd.DataFrame(results)

    results_df.to_csv(
        'mimic_first_25_results.csv',
        index=False
    )

    print('\nResults saved to mimic_first_25_results.csv')

    # Basic answer coverage evaluation
    valid_answers = results_df[
        results_df['Answer'] != 'Error'
    ]

    if len(results_df) > 0:
        answer_rate = len(valid_answers) / len(results_df)
    else:
        answer_rate = 0

    print(f'Answer generation rate: {answer_rate:.2%}')

else:
    results_df = pd.DataFrame()
    print('Evaluation skipped because QA dataset was not found.')

## 14. Display Performance

In [ ]:
if len(results_df) > 0:

    total = len(results_df)

    successful = len(
        results_df[results_df['Answer'] != 'Error']
    )

    success_rate = successful / total

    print('========== RAG PERFORMANCE ==========' )
    print('Total Questions :', total)
    print('Successful      :', successful)
    print('Answer Rate     :', f'{success_rate:.2%}')

    labels = ['Successful', 'Errors']
    values = [successful, total - successful]

    plt.figure(figsize=(8, 5))
    plt.bar(labels, values)
    plt.title('Clinical RAG Answer Generation Performance')
    plt.ylabel('Number of Questions')

    for i, value in enumerate(values):
        plt.text(i, value + 0.1, str(value), ha='center')

    plt.show()

else:
    print('No evaluation results available.')

print('\nClinical GenAI RAG pipeline completed successfully.')